<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.1B/MP8.1B_two_modules.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.1B · Two Modules on One Sink

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp81b.py', 'MP8.1B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.1B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp81b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp81b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP8.1B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Many converters carry two power modules on one heat sink - the two halves of a half bridge, or a rectifier beside an inverter. Where to place them is a design decision. Side by side they heat each other through the base plate; spread apart, each sits closer to an end of the sink, where the base has less metal to spread its heat. Somewhere between lies the **gap that makes the hottest module coolest**, and with it the most margin to the junction's limit.

On a real sink the answer depends on more than the conduction in the base: the fan blows the air across the fins, the air **warms** as it takes the heat up, and its coefficient **falls** as the boundary layer on the fins grows. A module downstream sits in warmer air, cooled less. The best gap is then no longer symmetric in its reasoning, and no longer the one a uniform-air calculation gives.

This mini project computes the hottest point of the two modules **for any gap from 0 to 40 mm** with one network, and then finds the **best gap by a gradient through the network**.

## A2 · The heat sink, and what is new against Ex08

The heat sink is Ex08's: a 100 mm wide, 10 mm thick aluminium base plate with ten fins collapsed into one effective coefficient on its underside, as in Ex08. The cell below draws it and prints its data.

**Two modules.** Ex08's 150 W module on 30 mm is split into two modules of 75 W on 15 mm each, the same flux of 50 kW/m², placed symmetrically about the base's centre with a gap $g$ between them, 0 to 40 mm.

**The air warms along the sink.** The fan blows the air across the fins in the direction of $x$, from $x = 0$ (the inlet) to $x = W$ (the outlet). What the fins give up at $x$ is carried downstream: the air's rise above the inlet, $\theta_a(x)$, obeys the air's energy balance

$$C\,\frac{d\theta_a}{dx} = S\,D\,h_\mathrm{eff}(x)\,\big(\theta(x, 0) - \theta_a(x)\big), \qquad \theta_a(0) = 0$$

with $C = \dot m c_p = 15$ W/K the air's heat capacity flow (about 0.015 kg/s; **assumed**, so that the air warms by 10 K across the sink at 150 W) and $D = 100$ mm the sink's depth.

**The coefficient falls along the sink.** The boundary layer on the fins grows along the flow, so $h$ is highest at the inlet. A linear fall is **assumed**: $h(x) = h_\mathrm{fan}\,\big(1 + S\,B\,(1 - 2x/W)\big)$ with $B = 0.25$: 50 W/(m²K) at the inlet and 30 at the outlet, the mean Ex08's 40. The collapsed coefficient $h_\mathrm{eff}(x)$ follows from Ex08's fin calculation.

$S$ is the **strength of the change**: at $S = 0$ the air keeps its inlet temperature and $h = 40$ everywhere, which is Ex08's problem with two footprints.

In [ ]:
# The heat sink with two modules, and its data
fig, ax = plt.subplots(figsize=(8, 3.6))
hs.draw_heatsink(ax, show_domain=True)                   # Ex08's drawing of the sink and the domain
for lo, hi, _ in mp.patches(0.0175):                      # the two modules at a 17.5 mm gap
    ax.add_patch(plt.Rectangle((lo * 1e3, hs.T_BASE * 1e3 + 7), (hi - lo) * 1e3, 4, color="tab:purple", alpha=0.6))
ax.annotate("", xy=(100, -55), xytext=(0, -55), arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2))
ax.text(50, -58, "air flows across the sink, warming", ha="center", va="top", color="tab:blue", fontsize=9)
ax.set_ylim(-65, 30); ax.set_title("Ex08's sink; MP8.1B's two modules (purple, 17.5 mm gap) and the air's path")
plt.show()
rows = [("base plate", f"{hs.W_BASE*1e3:.0f} × {hs.T_BASE*1e3:.0f} mm, k = {hs.K_AL:.0f} W/(m·K), depth {hs.DEPTH*1e3:.0f} mm"),
        ("modules", f"two of {mp.P_MODULE:.0f} W on {mp.A_MODULE*1e3:.0f} mm each, {mp.FLUX/1e3:.0f} kW/m²; gap {mp.G_MIN*1e3:.0f}–{mp.G_MAX*1e3:.0f} mm"),
        ("air", f"C = {mp.C_AIR:.0f} W/K, so it warms by {2*mp.P_MODULE/mp.C_AIR:.0f} K across the sink"),
        ("h on the fins", f"{mp.h_air(0.0):.0f} W/(m²K) at the inlet, {mp.h_air(hs.W_BASE):.0f} at the outlet"),
        ("h_eff on the base", f"{mp.h_eff(0.0):.0f} W/(m²K) at the inlet, {mp.h_eff(hs.W_BASE):.0f} at the outlet (Ex08: {hs.H_EFF:.0f})")]
for k, v in rows:
    print(f"{k:>17}  {v}")

**What you should see.** The sink with the two modules on top and the air's path below. The air warms by 10 K across the sink at 150 W. The collapsed coefficient falls from 393 W/(m²K) at the inlet to 248 at the outlet, about Ex08's 322 in the middle.

## A3 · The domain, the equations and their conditions

The domain is Ex08's base plate in cross-section, $0 < x < W$, $0 < z < t_b$, per metre of depth, with one more unknown along its underside: the air's rise $\theta_a(x)$. Both rises are measured above the air at the **inlet**.

$$\frac{\partial^2 \theta}{\partial x^2} + \frac{\partial^2 \theta}{\partial z^2} = 0 \quad \text{in the base plate}, \qquad C\,\frac{d\theta_a}{dx} = D\,h_\mathrm{eff}(x)\,(\theta - \theta_a) \quad \text{along } z = 0$$

| where | condition | why it holds |
|---|---|---|
| module face, on the two footprints ($z = t_b$) | $k\,\partial\theta/\partial z = q''$ | each module's 75 W enters on its 15 mm |
| module face, elsewhere | $\partial\theta/\partial z = 0$ | still air above the base |
| fin side ($z = 0$) | $k\,\partial\theta/\partial z = h_\mathrm{eff}(x)\,(\theta - \theta_a(x))$ | the fins give heat to the air at its local temperature, with the local coefficient |
| ends ($x = 0$, $x = W$) | $\partial\theta/\partial x = 0$ | insulated, as in Ex08 |
| the air at the inlet | $\theta_a(0) = 0$ | the rises are measured from the inlet air |

The two equations are **coupled**: the plate's underside sets the air's warming, and the air's temperature sets the underside's heat loss. Integrated over the sink, all the heat the modules put in ends in the air: $C\,\theta_a(W) = 150$ W, so $\theta_a(W) = 10$ K at the outlet whatever the gap. That is the energy balance of this problem.

## A4 · Why there is no exact solution

At $S = 0$ the problem is Ex08's with two footprints, and Ex08's series solves it exactly: the flux of two patches is a sum of cosines, and each cosine has its own exact profile through the thickness. With $S \neq 0$ the fin-side coefficient varies along $x$, so a cosine at the underside no longer stays one cosine: every mode is coupled to every other. And the air's temperature is itself an unknown, set by the plate's underside along the whole upstream path. Separation of variables fails, and no series or closed form exists.

The reference and the ground truth are therefore **numerical**: finite volumes for the plate, coupled to the air's balance cell by cell (part C). They are checked in three ways: on grids each twice as fine, by the energy balance, and at $S = 0$, where they must reproduce Ex08's exact series with two footprints.

In [ ]:
# The hottest point against the gap: with the air warming (this project) and with Ex08's air (S = 0)
G = np.load("MP8.1B_truth.npz")
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(G["gaps"] * 1e3, G["hottest"], label="air warming, h falling (MP8.1B)")
ax.plot(G["gaps"] * 1e3, G["hottest_S0_400"], "--", label="Ex08's air (S = 0)")
ax.axvline(float(G["best_gap"]) * 1e3, color="tab:blue", lw=0.6)
ax.set_xlabel("gap g [mm]"); ax.set_ylabel("hottest point above the inlet air [K]"); ax.legend()
plt.tight_layout(); plt.show()
i0 = int(np.argmin(G["hottest_S0_400"]))
print(f"MP8.1B: best gap {float(G['best_gap'])*1e3:.2f} mm, hottest {float(G['best_hottest']):.2f} K; "
      f"at 0 mm {G['hottest'][0]:.2f} K, at 40 mm {G['hottest'][-1]:.2f} K")
print(f"Ex08's air: best gap {G['gaps'][i0]*1e3:.1f} mm, hottest {G['hottest_S0_400'][i0]:.2f} K")

**What you should see.** Both curves fall as the modules move apart and rise again near 40 mm, where each module approaches an end of the base. With Ex08's air the best gap is 35 mm and the hottest point 49.22 K. With the air warming and the coefficient falling, the whole curve lies about 6.5 K higher - the downstream module sits in air up to 10 K warmer - and the best gap moves **closer**, to 28.17 mm, at 55.91 K: spreading the modules further pushes the downstream one into the warmest air at the weakest coefficient.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the hottest point of the module face | every gap from 0 to 40 mm, every 0.5 mm (81 gaps) | within **0.1 K** of the ground truth |
| the best gap, found by a gradient through the network | - | within **2 mm** of the ground truth's 28.17 mm |
| training | - | **one**: the gap is an input |
| the time to answer a new gap, and to find the best gap, online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures both) |

The best gap is the minimum of a flat curve: 0.1 K of error in the hottest point can move it by millimetres. That is why the gradient must come from a network that is accurate at every gap, not only at a few. The training time is reported apart, as the offline cost.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

Two unknowns, two networks (or one network with two outputs):

$$(x,\ z,\ g)\ \longmapsto\ \theta(x, z; g), \qquad (x,\ g)\ \longmapsto\ \theta_a(x; g)$$

the plate's rise above the inlet air at a point, for a gap, and the air's rise along the sink. The gap is an **input**: one training covers every gap, and the hottest point is a differentiable function of the gap, so its derivative with respect to $g$ comes from automatic differentiation, and the best gap from a search on it.

In [ ]:
# The scales of the problem
print(f"X = x / W, W = {hs.W_BASE*1e3:.0f} mm;  Z = z / t_b, t_b = {hs.T_BASE*1e3:.0f} mm;  gap 0 to {mp.G_MAX*1e3:.0f} mm")
print(f"residual scale S = q_mean t_b / k = {hs.Q_MEAN*hs.T_BASE/hs.K_AL:.2f} K  (the rise across the base's thickness)")
print(f"the air's rise at the outlet: 150 W / C = {2*mp.P_MODULE/mp.C_AIR:.1f} K; the hottest point is about 56 K")

**What you should see.** The air's rise (10 K) is a sixth of the hottest point (about 56 K), and the variation of the hottest point with the gap is 2 K over the whole range: the network has to be accurate to a small fraction of what it represents.

## B2 · The physics the network is trained with

| loss term | where it is evaluated | its residual, of order one |
|---|---|---|
| $\mathcal{L}_\mathrm{PDE}$ | points in the plate $(x, z, g)$ | $t_b^2(\theta_{xx} + \theta_{zz}) / S$ |
| $\mathcal{L}_\mathrm{fin}$ | points on the fin side $(x, g)$ | $t_b\,\big(\theta_z - h_\mathrm{eff}(x)(\theta - \theta_a)/k\big) / S$ |
| $\mathcal{L}_\mathrm{air}$ | the same points | $\big(C\,\theta_a' - D\,h_\mathrm{eff}(x)(\theta - \theta_a)\big)\,W / (150\ \mathrm{W})$ |
| $\mathcal{L}_\mathrm{module}$ | points on the module face | $t_b\,(\theta_z - q''(x)/k) / S$ |
| $\mathcal{L}_\mathrm{balance}$ (optional) | the outlet, at random gaps | $100\,(C\,\theta_a(W) / 150\ \mathrm{W} - 1)$, in per cent |

with $S = 0.9$ K the rise across the base's thickness.

**What can be built in rather than trained.** The insulated ends hold for any weights if the network sees $x$ only through $\cos(n\pi x/W)$. The air's inlet condition holds if $\theta_a$ is written as $x$ times a network. And the largest part of the answer is known exactly: **Ex08's series with the two footprints**, the solution at $S = 0$, which carries the flux of both modules for every gap. Its coefficients are $q_n(g)$, sines of the footprints' edges - differentiable in $g$. A network that adds to it a correction with zero slope at $z = t_b$ leaves the module face exact, and learns only what the warming air and the falling coefficient change: about 6.5 K, smooth in $x$, $z$ and $g$. `mp.ex08_series(x, z, g)` evaluates it in NumPy; part D has it in PyTorch.

## B3 · The training points

`mp81b.py` provides the samplers: `mp.sample_interior(n)` points $(x, z, g)$ in the plate, `mp.sample_fin_side(n)` points $(x, g)$ on the underside for the fin and air terms, and `mp.sample_top(n)` points $(x, g)$ on the module face, a fifth of them within 1.5 mm of a footprint's edge. Every point carries its own gap, uniform from 0 to 40 mm. The scoring gaps are every 0.5 mm, so any gap is a test gap: the network must be smooth in $g$ as well as accurate.

In [ ]:
# The training points, and the module face's flux at three gaps
x, z, g = mp.sample_interior(3000)
xt, gt = mp.sample_top(1000)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
ax[0].scatter(x * 1e3, z * 1e3, s=1, alpha=0.4); ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("z [mm]")
ax[1].scatter(xt * 1e3, gt * 1e3, s=1, alpha=0.5); ax[1].set_xlabel("x on the module face [mm]"); ax[1].set_ylabel("gap [mm]")
xs = np.linspace(0, hs.W_BASE, 1001)
for gg in (0.0, 0.0175, 0.040):
    ax[2].plot(xs * 1e3, mp.flux_top(xs, gg) / 1e3, label=f"g = {gg*1e3:.1f} mm")
ax[2].set_xlabel("x [mm]"); ax[2].set_ylabel("q'' [kW/m²]"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

**What you should see.** Left, points spread over the plate. Middle, the module-face points against their gap: denser along four slanted lines, the footprints' edges, which move apart as the gap grows. Right, the flux on the module face at three gaps: two blocks of 50 kW/m² that separate.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | finite volumes for one gap, and a one-dimensional search for the best gap | the same scheme on grids fine enough to be converged |
| grid | 50 × 5 cells of 2 mm, the coarsest that meets 0.1 K at every gap | 400 × 40 and 800 × 80 cells, extrapolated by Richardson |
| its job | to be **timed** against the network: a new gap, and the best gap | to **score** the network and the reference |

Both use `mp81b.Plate`.

## C2 · The scheme, step by step

1. **Cells.** $n_x \times n_z$ rectangular cells cover the base plate; the unknowns are the cells' rises and the air's rise at the $n_x + 1$ cell faces along the underside.
2. **Conduction.** Between two neighbouring cells the heat flow is $k$ times the difference of their rises, times the face's length over the centres' distance.
3. **The module face.** Each top cell receives the flux times the share of its face the footprints cover. A footprint's edge that falls inside a cell is shared exactly, so the scheme keeps its order for every gap - and the hottest point is a smooth function of the gap, which the 1-D search needs.
4. **The fin side, coupled to the air.** Each bottom cell passes $g_i\,(\theta_i - \bar\theta_{a,i})$ to the air, with $g_i$ half a cell of metal in series with $h_\mathrm{eff}(x_i)$ and $\bar\theta_{a,i}$ the mean of the air's rise at the cell's two faces. The air's rise steps from face to face by what the cell gives it, divided by $C$: the air's balance, cell by cell.
5. **One sparse solve.** Plate and air form one sparse linear system. Its matrix does not depend on the gap - only the right-hand side does - so it is factorised once, and a new gap is one back-substitution.
6. **The best gap.** The hottest point is read on the top face (the top cell plus half a cell of the flux's slope, a parabola through the hottest cell and its neighbours), and the best gap is found by a bounded 1-D search (Brent's method), one solve per trial gap.

In [ ]:
# The ground truth: convergence at a 17.5 mm gap, the balance, and Ex08's limit, from the truth file
G = np.load("MP8.1B_truth.npz")
print("cells (nx × nz)   hottest point     heat taken by the air - heat in")
for g, hot, b in zip(G["conv_grids"], G["conv_hottest"], G["conv_balance"]):
    print(f"{int(g[0]):5d} × {int(g[1]):3d}      {hot:10.4f} K      {b*150:+.1e} W")
d = np.abs(np.diff(G["conv_hottest"]))
print("change of the hottest between grids:", np.round(d, 5), "K")
print(f"Richardson extrapolation {float(G['conv_hottest_extrapolated']):.4f} K")
print(f"at S = 0, against Ex08's exact series with two footprints: {float(G['zero_strength_error_K']):.4f} K at most")
print(f"the scoring curve: 400 × 40 and 800 × 80 differ by {np.abs(G['hottest_800'] - G['hottest_400']).max():.4f} K at most over the 81 gaps")

**What you should see.** The hottest point changes by 3e-03, 4e-04 and 3e-04 K from grid to grid: the 2 mm grid is already within 0.004 K, and the finer grids change it by less than the parabola that reads the maximum between cells. The scoring curve is the Richardson extrapolation of the two finest grids, which differ by less than 3e-04 K: a thousand times finer than the 0.1 K asked. The air takes exactly the 150 W the modules put in, on every grid: the scheme conserves heat. At $S = 0$ the same code reproduces Ex08's exact series with two footprints to 0.002 K.

## C3 · The reference model: accuracy and time

The reference grid is 50 × 5 cells of 2 mm. The cell below scores it against the ground truth at all 81 gaps, finds its best gap, and times a new gap and a best-gap search. It runs on the CPU of the Colab runtime.

In [ ]:
# The reference model against the ground truth, at the 81 scoring gaps
ref = np.array([mp.reference(g)["hottest"] for g in G["gaps"]])
best = mp.reference_best_gap()
e_hot = np.abs(ref - G["hottest"]).max()
e_gap = abs(best["gap"] - float(G["best_gap"])) * 1e3
t_gap = middle(lambda: mp.reference(0.0175))             # the online cost of a new gap (matrix factorised once)
t_best = middle(lambda: mp.reference_best_gap())         # the online cost of the best gap
print(f"reference model: hottest point within {e_hot:.3f} K of the truth at every gap")
print(f"best gap {best['gap']*1e3:.2f} mm (truth {float(G['best_gap'])*1e3:.2f} mm, off by {e_gap:.2f} mm), {best['solves']} solves")
print(f"online: a new gap {t_gap*1e3:.3f} ms; the best gap {t_best*1e3:.2f} ms")
REF = {"t_gap_s": t_gap, "t_best_s": t_best, "worst_hot_K": e_hot, "gap_err_mm": e_gap, "best_gap_m": best["gap"]}   # kept for part E

**What you should see.** The 2 mm grid is within 0.03 K of the truth at every gap, and its best gap is 28.64 mm, 0.47 mm from the truth's 28.17 - both inside the requirement. A new gap takes a few hundredths of a millisecond (0.05 ms in the solved run), the best gap under a millisecond (ten solves, 0.62 ms). This is a very demanding bar: the matrix has 300 unknowns and is factorised once, so a new gap is one back-substitution. A network evaluated on a hundred points of the module face costs more than that on a CPU; part E measures both on this same runtime, and the solved run does **not** reach the ten times asked (see part E).

In [ ]:
# The ground truth at a 17.5 mm gap: the plate's field and the air along the sink
key = "17.5mm"
fig, ax = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw=dict(width_ratios=[2, 1]))
im = ax[0].imshow(G["field_" + key], origin="lower", aspect="auto", extent=[0, 100, 0, 10], cmap="inferno")
plt.colorbar(im, ax=ax[0], label="θ above the inlet air [K]"); ax[0].set_xlabel("x [mm]"); ax[0].set_ylabel("z [mm]")
ax[1].plot(G["x_faces"] * 1e3, G["air_" + key], label="air")
ax[1].plot(G["x"] * 1e3, G["field_" + key][0], label="plate's underside")
ax[1].set_xlabel("x [mm]"); ax[1].set_ylabel("rise above the inlet air [K]"); ax[1].legend()
plt.tight_layout(); plt.show()

**What you should see.** Left, the field at a 17.5 mm gap: the downstream module (right) is the hotter. Right, the air warms from 0 at the inlet to 10 K at the outlet, steepest under the modules; the plate's underside is about 50 K above the inlet air throughout, warmer downstream.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** (TODO 3).

**The contract.** Your network is a PyTorch module called `net` with two methods:

| call | meaning | returns |
|---|---|---|
| `net(x, z, g)` | the plate's rise at $(x, z)$ in metres for a gap $g$ in metres, each `(n, 1)` | $\theta$ in K above the inlet air, `(n, 1)` |
| `net.air(x, g)` | the air's rise at $x$ for a gap $g$ | $\theta_a$ in K, `(n, 1)` |

## D2 · The loss, given

`losses(net, seed)` draws fresh points with the samplers of part B and returns the four residual terms of part B's table; `balance_loss(net, seed)` is the energy balance at the outlet, at 64 random gaps. Every residual is scaled to be of order one.

`ex08_torch(x, z, g)` is Ex08's exact series with the two footprints in PyTorch - the solution at $S = 0$ - differentiable in the gap. Use it in your network if you choose (part B, section 2).

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    """du/dx by automatic differentiation, keeping the graph for a second derivative."""
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

K, TB, W, D = hs.K_AL, hs.T_BASE, hs.W_BASE, hs.DEPTH
S = hs.Q_MEAN * TB / K                                   # K, the residuals' scale (0.90 K)
P_IN = 2 * mp.P_MODULE                                   # W, the heat in

# Ex08's exact series with the two footprints (S = 0), in PyTorch and differentiable in the gap
NM = 100                                                 # cosine modes: the tail beyond 100 changes the hottest point by < 1e-03 K
LAM = torch.arange(NM, dtype=torch.float64, device=DEVICE) * np.pi / W
def qn(g):
    """cosine coefficients of the two footprints' flux, (n, NM): sines of the footprints' edges"""
    c, a = W / 2, mp.A_MODULE
    m = torch.arange(1, NM, dtype=torch.float64, device=DEVICE)[None, :]
    def ind(lo, hi):
        return torch.cat([(hi - lo) / W, 2.0 / (m * np.pi) * (torch.sin(m * np.pi * hi / W) - torch.sin(m * np.pi * lo / W))], 1)
    return mp.FLUX * (ind(c - g / 2 - a, c - g / 2) + ind(c + g / 2, c + g / 2 + a))

def ex08_torch(x, z, g):
    q = qn(g); l = LAM[1:][None, :]; r = hs.H_EFF / (K * l)
    num = (1 + r) * torch.exp(l * (z - TB)) + (1 - r) * torch.exp(-l * (z + TB))
    den = (1 + r) - (1 - r) * torch.exp(-2 * l * TB)
    return q[:, :1] * (z / K + 1 / hs.H_EFF) + (q[:, 1:] / (K * l) * num / den * torch.cos(l * x)).sum(1, keepdim=True)

def heff(x):
    """h_eff(x) (W/(m²K)) at the points x, a constant of the problem"""
    return T(mp.h_eff(x.detach().cpu().numpy().ravel()))

def losses(net, seed, n_in=1000, n_face=500):
    # in the plate: θ_xx + θ_zz = 0
    x, z, g = map(T, mp.sample_interior(n_in, seed))
    x.requires_grad_(True); z.requires_grad_(True)       # autograd differentiates with respect to these
    th = net(x, z, g)
    r_pde = TB ** 2 * (d(d(th, x), x) + d(d(th, z), z)) / S
    # the fin side, z = 0, and the air along it
    xb, gb = map(T, mp.sample_fin_side(n_face, seed + 1))
    xb.requires_grad_(True); zb = torch.zeros_like(xb, requires_grad=True)
    tb, ta, he = net(xb, zb, gb), net.air(xb, gb), heff(xb)
    r_fin = TB * (d(tb, zb) - he / K * (tb - ta)) / S
    r_air = (mp.C_AIR * d(ta, xb) - D * he * (tb - ta)) * W / P_IN
    # the module face, z = t_b: θ_z = q''(x) / k
    xt, gt = mp.sample_top(n_face, seed + 2)
    zt = T(np.full(len(xt), TB)).requires_grad_(True)
    tt = net(T(xt), zt, T(gt))
    r_top = TB * (d(tt, zt) - T(mp.flux_top(xt, gt)) / K) / S
    return {"pde": (r_pde ** 2).mean(), "fin": (r_fin ** 2).mean(), "air": (r_air ** 2).mean(), "module": (r_top ** 2).mean()}

def balance_loss(net, seed, n=64):
    # all the heat ends in the air: C θ_a(W) = 150 W, in per cent
    g = T(np.random.default_rng(seed).uniform(mp.G_MIN, mp.G_MAX, n))
    return ((mp.C_AIR * net.air(torch.full_like(g, W), g) / P_IN - 1) * 100).pow(2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` with `forward(x, z, g)` and `air(x, g)`, and make `net = Net().to(DEVICE)`. Decide what the networks see (scaled inputs, cosines of $x$), what is built in (the insulated ends, the inlet air, Ex08's series as the starting answer), and their size. The hottest point varies by 2 K over the range of gaps and must be right to 0.1 K at every gap: the network must be smooth in $g$.

In [ ]:
# The hottest point from the network, and a quick score, given
G = np.load("MP8.1B_truth.npz")
XT = T(np.arange(mp.X_HOT[0], mp.X_HOT[1] + 1e-9, 0.0005))   # m, the module face every 0.5 mm, where the hottest point lies
def hottest(net, g):
    """the hottest point of the module face (K) for a batch of gaps g (m), shape (ng,); differentiable in g"""
    g = g.reshape(-1, 1); ng, nx = g.shape[0], XT.shape[0]
    th = net(XT.repeat(ng, 1), torch.full((ng * nx, 1), TB, device=DEVICE), g.repeat_interleave(nx, 0)).reshape(ng, nx)
    i = th.argmax(1).clamp(1, nx - 2)
    a, b, c = (th.gather(1, (i + k)[:, None]) for k in (-1, 0, 1))
    return (b - (a - c) ** 2 / (8 * (a - 2 * b + c))).ravel()   # the top of the parabola through three points

def quick_score():
    with torch.no_grad():
        return float(np.abs(hottest(net, T(G["gaps"])).cpu().numpy() - G["hottest"]).max())

In [ ]:
# TODO 1 - your network: net(x, z, g) -> θ (K), net.air(x, g) -> θ_a (K), every argument an (n, 1) tensor
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, x, z, g):
        ...                                              # the plate's rise, shape (n, 1)

    def air(self, x, g):
        ...                                              # the air's rise, shape (n, 1)

torch.manual_seed(81)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` with `losses` (and, if you choose, `balance_loss`). Keep the history of each term for TODO 3, and **measure the training time** in `TRAIN_SECONDS`.

Given below: `hottest(net, g)`, the network's hottest point for a batch of gaps (the module face every 0.5 mm from 45 to 95 mm, where the hottest point lies at every gap, and a parabola through the largest value and its neighbours), and `quick_score()`, the worst error of the hottest point over the 81 scoring gaps.

In [ ]:
# TODO 2 - your training
history = {"pde": [], "fin": [], "air": [], "module": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst error of the hottest point over the gaps {quick_score():.3f} K")

## D5 · TODO 3 · Your figures, and the best gap

Draw the loss terms during training and the hottest point against the gap from your network and the ground truth, with its error. Then **find the best gap by a gradient through the network**: write `best_gap()`, which returns the gap (m) that minimises `hottest(net, g)`, using `torch.autograd.grad` of the hottest point with respect to `g`. The evaluation times it. Save figures as `MP8.1B_report_<name>.png`.

In [ ]:
# TODO 3 - your figures, and the best gap by a gradient through the network
def best_gap():
    ...                                                  # return the gap in metres

...                                                      # your figures

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores and keeps the answers in `R`: the hottest point at the 81 scoring gaps, the best gap from your `best_gap()`, the air's rise at the outlet at those gaps, and the **online times**: one new gap, the best gap, and 81 gaps in one call, against the reference model on the same runtime.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {}
with torch.no_grad():
    out["hottest"] = hottest(net, T(G["gaps"])).cpu().numpy()
    out["air_out"] = net.air(torch.full((G["gaps"].size, 1), W, device=DEVICE), T(G["gaps"])).cpu().numpy().ravel()
out["best_gap"] = np.array(best_gap())
timing = {}
def one_gap():
    with torch.no_grad():
        return hottest(net, T([0.0175]))
def all_gaps():
    with torch.no_grad():
        return hottest(net, T(G["gaps"]))
timing["net_gap"] = middle(one_gap)
timing["net_best"] = middle(best_gap)
timing["net_81"] = middle(all_gaps)
timing["ref_gap"] = middle(lambda: mp.reference(0.0175))
timing["ref_best"] = middle(lambda: mp.reference_best_gap())
timing["ref_81"] = middle(lambda: [mp.reference(g) for g in G["gaps"]])
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP8.1B_net.pt"))
print(f"best gap: network {float(out['best_gap'])*1e3:.2f} mm, truth {float(G['best_gap'])*1e3:.2f} mm")
for k in ("gap", "best", "81"):
    print(f"online, {k:>4}: network {timing['net_' + k]*1e3:7.3f} ms, reference {timing['ref_' + k]*1e3:7.3f} ms "
          f"(ratio {timing['ref_' + k] / timing['net_' + k]:.2f})")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model's times `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP8.1B_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): hottest within {REF['worst_hot_K']:.3f} K, best gap off by {REF['gap_err_mm']:.2f} mm; "
      f"{REF['t_gap_s']*1e3:.3f} ms per gap, {REF['t_best_s']*1e3:.2f} ms per best gap")

## E2 · TODO 4 · The goal metrics

Compute and put in `METRICS`: the worst error of the hottest point over the 81 gaps (K), the error of the best gap (mm), the air's rise at the outlet against 10 K (largest error, K), the online times of a new gap, of the best gap and of 81 gaps for the network and the reference, their ratios, and the training time. Say in one line each whether the requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw and save as `MP8.1B_report_<name>.png`: the hottest point against the gap from the network, the reference model, the ground truth and Ex08's air; the error of each against the truth; and the online times.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 10 are about your network and its result; 11 to 17 ask you to use it; 18 concludes. The next cell writes `MP8.1B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (with any features), layers, neurons and number of weights, for the plate and for the air. Why this size?": """
""",
    "2. Which conditions did you build into the network's form, and which into the loss? Give each loss term's value at the end of training.": """
""",
    "3. Did you build Ex08's series with two footprints into your network? Give the worst error of the hottest point before training and after.": """
""",
    "4. Your training: optimiser(s), steps, learning rates, number of points of each kind, and the offline time in seconds.": """
""",
    "5. How did you weight the loss terms, the energy balance among them? Show the terms during training and say which was hardest to bring down.": """
""",
    "6. The hottest point over the 81 gaps: your worst error, and at which gap. Is it within 0.1 K everywhere? Compare with the reference model's.": """
""",
    "7. Your best gap by the gradient against the truth's 28.17 mm. How did you search, how many gradient evaluations, and how far off is it?": """
""",
    "8. The air's rise at the outlet from your network against the 10 K the balance requires, at every gap. How far from closing?": """
""",
    "9. The online time for a new gap and for the best gap: your network against the reference model on the same runtime, and the ratios. What does each spend its time on?": """
""",
    "10. Ask for all 81 gaps at once: give both times and say why the ratio differs from the ratio for one gap.": """
""",
    "11. By how much does the air warming and h falling raise the hottest point at the best gap, against Ex08's air (S = 0)? Give both values.": """
""",
    "12. Why does the best gap move from 35 mm with Ex08's air to about 28 mm here? Use your network's field at the two gaps to explain.": """
""",
    "13. Which module is the hotter, upstream or downstream, and by how much at a 20 mm gap? Give both modules' hottest points from your network.": """
""",
    "14. If the fan ran slower (C = 10 W/K instead of 15), would the best gap move closer or further? Argue with your network or with the reference model.": """
""",
    "15. Train again with another seed. How much do the worst error and the best gap of the two runs differ? Is the best gap reproducible within 2 mm?": """
""",
    "16. How flat is the curve near the optimum? Give the range of gaps within 0.1 K of the best hottest point, and say what it means for the tolerance of the assembly.": """
""",
    "17. If the two modules carried different powers (60 and 90 W), could your network answer? What would you change to make it answer any split?": """
""",
    "18. To conclude: for which engineering task would you use your network rather than the reference model, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.3f} |" for k, v in METRICS.items()]
    out = ["# MP8.1B — Two Modules on One Sink", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP8.1B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP8.1B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP8.1B_report.md into MP8.1B_report.pdf, with any figure
# saved as MP8.1B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP8.1B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP8.1B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP8.1B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP8.1B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass